In [1]:
%cd /home/brimmann/works/xRAG

/home/brimmann/works/xRAG


/home/brimmann/works/xRAG/.venv/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [1]:
import torch

In [3]:
model_name = "Hannibal046/xrag-7b"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [4]:
from transformers import (
    AutoTokenizer
)

In [5]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    padding_side = 'left',
    add_eos_token=False, ## import to include this!
    use_fast=False,
)

In [6]:
from transformers import AutoModelForCausalLM
from transformers import AutoConfig
config = AutoConfig.from_pretrained(model_name)
from src.model import XMistralForCausalLM

In [7]:
 ## load llm
config = AutoConfig.from_pretrained(model_name)
MODEL_CLASS = eval(config.architectures[0])
model = MODEL_CLASS.from_pretrained(
    model_name,
    torch_dtype = torch.bfloat16,
    low_cpu_mem_usage = True,
    device_map='auto',
    offload_folder="./offload"
)
model.eval()
xrag_token = "<xRAG>"
assert xrag_token in tokenizer.get_vocab() 
model.set_xrag_token_id(tokenizer.convert_tokens_to_ids(xrag_token))

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [24]:
docs_embeds = torch.load("tensorstorage/doc_embeddings.pt").to(device)
relevant_doc = docs_embeds[0, :]
question = """What company advertised itself with the slogan "We'll leave a light on for you"?"""
## 4. concate the doc and query in a template
rag_template = """[INST] Refer to the background and rewrite it:

Background: {document}

[/INST]"""
prompt = rag_template.format_map(dict(document=xrag_token))
print(prompt)
prompt = rag_template.format_map(dict(document=xrag_token))
input_ids = tokenizer(prompt,return_tensors='pt').input_ids.to(device)

[INST] Refer to the background and rewrite it:

Background: <xRAG>

[/INST]


In [25]:
relevant_doc.unsqueeze(0).shape

torch.Size([1, 4096])

In [26]:
generated_output = model.generate(
        input_ids = input_ids,
        do_sample=False,
        max_new_tokens=20,
        pad_token_id=tokenizer.pad_token_id,
        retrieval_embeds = relevant_doc.unsqueeze(0),
    )
result = tokenizer.batch_decode(generated_output,skip_special_tokens=True)[0]
print(result)

Shape of final inputs_embeds fed to the model: torch.Size([1, 26, 4096])
The Alvin and the Chipmunks are a fictional singing group consisting of three chip


### XGemma2

In [2]:
import torch
from transformers import AutoTokenizer
from src.distill.models.modeling_xgemma import XGemmaForCausalLM, XGemmaConfig

# Specify model and device
model_name = "google/gemma-2-2b"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
# Load Tokenizer and add the special XRAG token
tokenizer = AutoTokenizer.from_pretrained(model_name)
xrag_token = "<xRAG>"
tokenizer.add_special_tokens({"additional_special_tokens": [xrag_token]})
xrag_token_id = tokenizer.convert_tokens_to_ids(xrag_token)


In [4]:
retriever_hidden_size = 4096
config = XGemmaConfig.from_pretrained(
    model_name,
    projector_type='mlp2x_gelu',
    retriever_hidden_size=retriever_hidden_size,
)

In [5]:
# Create a config with our custom parameters


# Load the model with this config
model = XGemmaForCausalLM.from_pretrained(model_name, config=config)

# Resize token embeddings layer to account for the new special token
model.resize_token_embeddings(len(tokenizer))

# Set the special token ID on the model instance and move to device
model.set_xrag_token_id(xrag_token_id)
model.to(device)
model.eval()

print(f"Model {model_name} loaded successfully onto {device}.")
print(f"Projector is configured: {hasattr(model, 'projector')}")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of XGemmaForCausalLM were not initialized from the model checkpoint at google/gemma-2-2b and are newly initialized: ['projector.projector.0.bias', 'projector.projector.0.weight', 'projector.projector.2.bias', 'projector.projector.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model google/gemma-2-2b loaded successfully onto cuda.
Projector is configured: True


In [6]:
model.projector.state_dict()

OrderedDict([('projector.0.weight',
              tensor([[-0.0273,  0.0144, -0.0194,  ...,  0.0305, -0.0169,  0.0002],
                      [ 0.0093, -0.0059,  0.0162,  ...,  0.0054,  0.0117, -0.0224],
                      [-0.0122, -0.0137, -0.0112,  ..., -0.0026,  0.0231,  0.0026],
                      ...,
                      [ 0.0066, -0.0182, -0.0047,  ..., -0.0162,  0.0010,  0.0232],
                      [-0.0067, -0.0093, -0.0083,  ..., -0.0111, -0.0177,  0.0260],
                      [-0.0224,  0.0178, -0.0073,  ..., -0.0412,  0.0466,  0.0143]],
                     device='cuda:0')),
             ('projector.0.bias',
              tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0')),
             ('projector.2.weight',
              tensor([[ 0.0113,  0.0161, -0.0293,  ...,  0.0099,  0.0145,  0.0030],
                      [-0.0435,  0.0092, -0.0251,  ..., -0.0126, -0.0117,  0.0068],
                      [ 0.0548, -0.0031, -0.0106,  ..., -0.0225, -0.0221, -0.0026]

In [7]:
u = torch.load("tensorstorage/best_student_projector.pth")

In [8]:
u

OrderedDict([('projector.0.weight',
              tensor([[-0.0783,  0.1009, -0.0589,  ..., -0.0072,  0.0125, -0.0059],
                      [ 0.0661, -0.0442, -0.0551,  ..., -0.0165, -0.0299,  0.0614],
                      [-0.0058,  0.0217, -0.0348,  ..., -0.0133,  0.0003, -0.0906],
                      ...,
                      [ 0.0562,  0.0313, -0.0294,  ..., -0.0268,  0.0012,  0.0183],
                      [ 0.1301,  0.0398, -0.0234,  ..., -0.0089,  0.0028, -0.0616],
                      [-0.0331,  0.0618, -0.0189,  ...,  0.0324,  0.0360,  0.0024]],
                     device='cuda:0')),
             ('projector.0.bias',
              tensor([ 0.0279,  0.0030, -0.0064,  ...,  0.0045,  0.0160,  0.0156],
                     device='cuda:0')),
             ('projector.2.weight',
              tensor([[-0.0158,  0.0450,  0.0162,  ..., -0.0496, -0.0381,  0.0459],
                      [-0.0211,  0.0381,  0.0423,  ...,  0.0140,  0.0097,  0.0358],
                      [-0.0190,

In [9]:
model.projector.load_state_dict(u)

<All keys matched successfully>

In [10]:
model.projector.state_dict()

OrderedDict([('projector.0.weight',
              tensor([[-0.0783,  0.1009, -0.0589,  ..., -0.0072,  0.0125, -0.0059],
                      [ 0.0661, -0.0442, -0.0551,  ..., -0.0165, -0.0299,  0.0614],
                      [-0.0058,  0.0217, -0.0348,  ..., -0.0133,  0.0003, -0.0906],
                      ...,
                      [ 0.0562,  0.0313, -0.0294,  ..., -0.0268,  0.0012,  0.0183],
                      [ 0.1301,  0.0398, -0.0234,  ..., -0.0089,  0.0028, -0.0616],
                      [-0.0331,  0.0618, -0.0189,  ...,  0.0324,  0.0360,  0.0024]],
                     device='cuda:0')),
             ('projector.0.bias',
              tensor([ 0.0279,  0.0030, -0.0064,  ...,  0.0045,  0.0160,  0.0156],
                     device='cuda:0')),
             ('projector.2.weight',
              tensor([[-0.0158,  0.0450,  0.0162,  ..., -0.0496, -0.0381,  0.0459],
                      [-0.0211,  0.0381,  0.0423,  ...,  0.0140,  0.0097,  0.0358],
                      [-0.0190,

In [10]:
p2 = model.projector.state_dict()

In [11]:
# --- Debugging Cell ---

# 1. Load the state dict from the file into a separate variable first
loaded_state_dict = torch.load("tensorstorage/best_student_projector.pth")

# 2. Check for differences between the initial weights (p1) and the file's weights
print("--- Comparing initial weights (p1) with the loaded file ---")
are_different_from_file = any(not torch.equal(p1[key], loaded_state_dict[key].to(p1[key].device)) for key in p1)
if are_different_from_file:
    print("✅ The initial weights and the file weights ARE different.")
else:
    print("❌ The initial weights and the file weights are THE SAME. This is likely the root cause.")

# 3. Print out some tensor values to manually inspect
print("\n--- Manual Inspection of Tensors ---")
# Get a key from one of the linear layers in the projector
sample_key = 'projector.0.weight' 

if sample_key in p1 and sample_key in p2 and sample_key in loaded_state_dict:
    print(f"A few values from '{sample_key}':")
    # Print the first 5 values from the flattened tensor for each state
    print("Initial (p1):      ", p1[sample_key].flatten()[:5])
    print("From File:         ", loaded_state_dict[sample_key].flatten()[:5])
    print("After load (p2):   ", p2[sample_key].flatten()[:5])
else:
    print(f"Could not find sample key '{sample_key}' in one of the state dicts.")
    print("Available keys in p1:", list(p1.keys()))


--- Comparing initial weights (p1) with the loaded file ---
❌ The initial weights and the file weights are THE SAME. This is likely the root cause.

--- Manual Inspection of Tensors ---
A few values from 'projector.0.weight':
Initial (p1):       tensor([-0.0783,  0.1009, -0.0589, -0.0308,  0.0479], device='cuda:0')
From File:          tensor([-0.0783,  0.1009, -0.0589, -0.0308,  0.0479], device='cuda:0')
After load (p2):    tensor([-0.0783,  0.1009, -0.0589, -0.0308,  0.0479], device='cuda:0')


In [13]:
model.eval()

XGemmaForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256001, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemma2RMSNo

In [14]:
# For a quick boolean check, you can do this:
are_different = any(not torch.equal(p1[key], p2[key]) for key in p1)
if are_different:
    print("\nVerification successful: The state dictionaries are different.")
else:
    print("\nVerification failed: The state dictionaries are the same.")


Verification failed: The state dictionaries are the same.


In [11]:
docs_embeds = torch.load("tensorstorage/doc_embeddings.pt").to(device)
relevant_doc = docs_embeds[0, :]
rag_template = """<start_of_turn>user
Refer to the background and rewrite it:

Background: {document}<end_of_turn>
<start_of_turn>model"""
prompt = rag_template.format_map(dict(document=xrag_token))
print(prompt)
prompt = rag_template.format_map(dict(document=xrag_token))
input_ids = tokenizer(prompt,return_tensors='pt').input_ids.to(device)

<start_of_turn>user
Refer to the background and rewrite it:

Background: <xRAG><end_of_turn>
<start_of_turn>model


In [12]:
generated_output = model.generate(
        input_ids = input_ids,
        do_sample=False,
        max_new_tokens=20,
        pad_token_id=tokenizer.pad_token_id,
        retrieval_embeds = relevant_doc.unsqueeze(0),
    )
result = tokenizer.batch_decode(generated_output,skip_special_tokens=True)[0]
print(result)


Refer to the background and rewrite it:

Background: StoryboardSegue
StoryboardSegue
StoryboardSegue
StoryboardSegue
